In [2]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout,
    LSTM,
    Bidirectional
)

from tensorflow.keras.callbacks import EarlyStopping

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)
print("CNN and RNN environment ready!")

TensorFlow: 2.21.0
GPU available: False
CNN and RNN environment ready!


In [3]:

from pathlib import Path

DATA_PATH = Path("../Dataset")
RESULT_PATH = Path("../Results")
MODEL_PATH = Path("../Models")

RESULT_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)

response_df = pd.read_json(
    DATA_PATH / "response.jsonl",
    lines=True
)

source_df = pd.read_json(
    DATA_PATH / "source_info.jsonl",
    lines=True
)

combined_df = response_df.merge(
    source_df,
    on="source_id",
    how="left"
)

combined_df["target"] = combined_df["labels"].apply(
    lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0
)

model_df = combined_df[
    combined_df["quality"] == "good"
].copy()

print("Dataset shape:", model_df.shape)
print("\nTarget distribution:")
print(model_df["target"].value_counts())
print("\nSplit distribution:")
print(model_df["split"].value_counts())


Dataset shape: (17617, 13)

Target distribution:
target
0    9976
1    7641
Name: count, dtype: int64

Split distribution:
split
train    14942
test      2675
Name: count, dtype: int64


In [4]:

original_train = model_df[
    model_df["split"] == "train"
].copy()

test_df = model_df[
    model_df["split"] == "test"
].copy()

train_df, val_df = train_test_split(
    original_train,
    test_size=0.10,
    random_state=42,
    stratify=original_train["target"]
)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Testing examples:", len(test_df))


Training examples: 13447
Validation examples: 1495
Testing examples: 2675


In [5]:

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

for df in [train_df, val_df, test_df]:
    df["source_info"] = df["source_info"].apply(clean_text)
    df["response"] = df["response"].apply(clean_text)

X_train = (
    train_df["source_info"] + " [SEP] " + train_df["response"]
).tolist()

X_val = (
    val_df["source_info"] + " [SEP] " + val_df["response"]
).tolist()

X_test = (
    test_df["source_info"] + " [SEP] " + test_df["response"]
).tolist()

y_train = train_df["target"].values
y_val = val_df["target"].values
y_test = test_df["target"].values

print("Text preparation completed.")


Text preparation completed.


In [6]:

MAX_WORDS = 20000
MAX_LEN = 512

keras_tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

keras_tokenizer.fit_on_texts(X_train)

train_sequences = keras_tokenizer.texts_to_sequences(X_train)
val_sequences = keras_tokenizer.texts_to_sequences(X_val)
test_sequences = keras_tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    train_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    val_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    test_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

vocab_size = min(
    MAX_WORDS,
    len(keras_tokenizer.word_index) + 1
)

print("Vocabulary size:", vocab_size)
print("Training shape:", X_train_pad.shape)
print("Validation shape:", X_val_pad.shape)
print("Testing shape:", X_test_pad.shape)


Vocabulary size: 20000
Training shape: (13447, 512)
Validation shape: (1495, 512)
Testing shape: (2675, 512)


In [7]:
# Check prepared input and label variables

for name, value in globals().copy().items():
    if name.startswith(("X_", "y_")):
        if hasattr(value, "shape"):
            print(name, ":", value.shape)

y_train : (13447,)
y_val : (1495,)
y_test : (2675,)
X_train_pad : (13447, 512)
X_val_pad : (1495, 512)
X_test_pad : (2675, 512)


In [8]:

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Conv1D, GlobalMaxPooling1D,
    Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Create folders
os.makedirs("Models/CNN", exist_ok=True)
os.makedirs("Results/CNN", exist_ok=True)

# Model configuration
VOCAB_SIZE = 20000
MAX_LENGTH = 512

# CNN architecture
cnn_model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),

    SpatialDropout1D(0.2),

    Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu",
        padding="valid"
    ),

    GlobalMaxPooling1D(),

    Dense(64, activation="relu"),
    Dropout(0.4),

    Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

# Callbacks
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        "Models/CNN/best_cnn.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

# Train CNN
cnn_history = cnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# Test evaluation
cnn_probabilities = cnn_model.predict(
    X_test_pad,
    batch_size=64,
    verbose=1
).ravel()

cnn_predictions = (cnn_probabilities >= 0.5).astype(int)

# Metrics
cnn_results = {
    "Model": "CNN",
    "Accuracy": accuracy_score(y_test, cnn_predictions),
    "Hallucination_Precision": precision_score(
        y_test, cnn_predictions, zero_division=0
    ),
    "Hallucination_Recall": recall_score(
        y_test, cnn_predictions, zero_division=0
    ),
    "Hallucination_F1": f1_score(
        y_test, cnn_predictions, zero_division=0
    )
}

print("\nCNN TEST RESULTS")
for metric, value in cnn_results.items():
    print(f"{metric}: {value}" if isinstance(value, float)
          else f"{metric}: {value}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    cnn_predictions,
    target_names=["Supported", "Hallucinated"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, cnn_predictions))

# Save model and results
cnn_model.save("Models/CNN/cnn_final.keras")

pd.DataFrame([cnn_results]).to_csv(
    "Results/CNN/cnn_results.csv",
    index=False
)

pd.DataFrame(cnn_history.history).to_csv(
    "Results/CNN/cnn_training_history.csv",
    index=False
)

print("\nCNN model and results saved successfully!")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 47s 209ms/step - accuracy: 0.6973 - loss: 0.5880 - val_accuracy: 0.7358 - val_loss: 0.5383
Epoch 2/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 99s 288ms/step - accuracy: 0.7596 - loss: 0.5108 - val_accuracy: 0.7391 - val_loss: 0.5357
Epoch 3/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 60s 286ms/step - accuracy: 0.8081 - loss: 0.4348 - val_accuracy: 0.7358 - val_loss: 0.5665
Epoch 4/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 62s 291ms/step - accuracy: 0.8560 - loss: 0.3420 - val_accuracy: 0.7264 - val_loss: 0.6615
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step

CNN TEST RESULTS
Model: CNN
Accuracy: 0.7842990654205607
Hallucination_Precision: 0.7466307277628033
Hallucination_Recall: 0.5874867444326617
Hallucination_F1: 0.657566765578635

Classification Report:
              precision    recall  f1-score   support

   Supported       0.80      0.89      0.84      1732
Hallucinated       0.75      0.59      0.66       943

    accuracy                           0.78      2675
   macro avg       

In [9]:

import os
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, GRU, Bidirectional,
    Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Create folders
os.makedirs("Models/RNN", exist_ok=True)
os.makedirs("Results/RNN", exist_ok=True)

# RNN architecture: Bidirectional GRU
rnn_model = Sequential([
    Embedding(
        input_dim=20000,
        output_dim=64,
        input_length=512
    ),

    SpatialDropout1D(0.2),

    Bidirectional(
        GRU(64)
    ),

    Dense(64, activation="relu"),
    Dropout(0.4),

    Dense(1, activation="sigmoid")
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_model.summary()

# Callbacks
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        "Models/RNN/best_rnn.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

# Train RNN
rnn_history = rnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=4,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# Test evaluation
rnn_probabilities = rnn_model.predict(
    X_test_pad,
    batch_size=64,
    verbose=1
).ravel()

rnn_predictions = (rnn_probabilities >= 0.5).astype(int)

# Calculate metrics
rnn_results = {
    "Model": "Bidirectional GRU",
    "Accuracy": accuracy_score(y_test, rnn_predictions),
    "Hallucination_Precision": precision_score(
        y_test, rnn_predictions, zero_division=0
    ),
    "Hallucination_Recall": recall_score(
        y_test, rnn_predictions, zero_division=0
    ),
    "Hallucination_F1": f1_score(
        y_test, rnn_predictions, zero_division=0
    )
}

print("\nRNN TEST RESULTS")

for metric, value in rnn_results.items():
    print(f"{metric}: {value}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    rnn_predictions,
    target_names=["Supported", "Hallucinated"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rnn_predictions))

# Save model
rnn_model.save("Models/RNN/rnn_final.keras")

# Save results
pd.DataFrame([rnn_results]).to_csv(
    "Results/RNN/rnn_results.csv",
    index=False
)

pd.DataFrame(rnn_history.history).to_csv(
    "Results/RNN/rnn_training_history.csv",
    index=False
)

print("\nRNN model and results saved successfully!")


c:\Users\Yashi\tf_env\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/4
211/211 ━━━━━━━━━━━━━━━━━━━━ 285s 1s/step - accuracy: 0.6903 - loss: 0.5900 - val_accuracy: 0.7445 - val_loss: 0.5371
Epoch 2/4
211/211 ━━━━━━━━━━━━━━━━━━━━ 268s 1s/step - accuracy: 0.7614 - loss: 0.5092 - val_accuracy: 0.7378 - val_loss: 0.5456
Epoch 3/4
211/211 ━━━━━━━━━━━━━━━━━━━━ 221s 1s/step - accuracy: 0.7895 - loss: 0.4703 - val_accuracy: 0.7264 - val_loss: 0.5958
42/42 ━━━━━━━━━━━━━━━━━━━━ 11s 235ms/step

RNN TEST RESULTS
Model: Bidirectional GRU
Accuracy: 0.7854205607476635
Hallucination_Precision: 0.7971014492753623
Hallucination_Recall: 0.5249204665959704
Hallucination_F1: 0.6329923273657289

Classification Report:
              precision    recall  f1-score   support

   Supported       0.78      0.93      0.85      1732
Hallucinated       0.80      0.52      0.63       943

    accuracy                           0.79      2675
   macro avg       0.79      0.73      0.74      2675
weighted avg       0.79      0.79      0.77      2675

Confusion Matrix:
[[1606  126

In [10]:

import os
import pandas as pd

# Create main Results folder
os.makedirs("Results", exist_ok=True)

# Results from all models
all_results = [
    {
        "Model": "Logistic Regression",
        "Accuracy": 0.7686,
        "Hallucination_Precision": 0.70,
        "Hallucination_Recall": 0.60,
        "Hallucination_F1": 0.65
    },
    {
        "Model": "Naive Bayes",
        "Accuracy": 0.7439,
        "Hallucination_Precision": 0.64,
        "Hallucination_Recall": 0.61,
        "Hallucination_F1": 0.63
    },
    {
        "Model": "SVM",
        "Accuracy": 0.7779,
        "Hallucination_Precision": 0.74,
        "Hallucination_Recall": 0.57,
        "Hallucination_F1": 0.64
    },
    {
        "Model": "Random Forest",
        "Accuracy": 0.7809,
        "Hallucination_Precision": 0.74,
        "Hallucination_Recall": 0.59,
        "Hallucination_F1": 0.65
    },
    {
        "Model": "ANN",
        "Accuracy": 0.7058,
        "Hallucination_Precision": 0.57,
        "Hallucination_Recall": 0.69,
        "Hallucination_F1": 0.62
    },
    {
        "Model": "CNN",
        "Accuracy": cnn_results["Accuracy"],
        "Hallucination_Precision": cnn_results["Hallucination_Precision"],
        "Hallucination_Recall": cnn_results["Hallucination_Recall"],
        "Hallucination_F1": cnn_results["Hallucination_F1"]
    },
    {
        "Model": "Bidirectional GRU",
        "Accuracy": rnn_results["Accuracy"],
        "Hallucination_Precision": rnn_results["Hallucination_Precision"],
        "Hallucination_Recall": rnn_results["Hallucination_Recall"],
        "Hallucination_F1": rnn_results["Hallucination_F1"]
    }
]

comparison_df = pd.DataFrame(all_results)

# Sort by hallucination F1
comparison_df = comparison_df.sort_values(
    by="Hallucination_F1",
    ascending=False
).reset_index(drop=True)

# Save results
comparison_df.to_csv(
    "Results/model_comparison.csv",
    index=False
)

# Display percentage-based comparison
display(
    comparison_df.style.format({
        "Accuracy": "{:.2%}",
        "Hallucination_Precision": "{:.2%}",
        "Hallucination_Recall": "{:.2%}",
        "Hallucination_F1": "{:.2%}"
    })
)

print("Master comparison saved successfully!")


,Model,Accuracy,Hallucination_Precision,Hallucination_Recall,Hallucination_F1
0,CNN,78.43%,74.66%,58.75%,65.76%
1,Random Forest,78.09%,74.00%,59.00%,65.00%
2,Logistic Regression,76.86%,70.00%,60.00%,65.00%
3,SVM,77.79%,74.00%,57.00%,64.00%
4,Bidirectional GRU,78.54%,79.71%,52.49%,63.30%
5,Naive Bayes,74.39%,64.00%,61.00%,63.00%
6,ANN,70.58%,57.00%,69.00%,62.00%


Master comparison saved successfully!
